# Spatial Search on 3D quantum walk

This notebook is used for any arbital edge lenth $x$ of $N=x^2$.

In [306]:
#import classiq 
#classiq.authenticate()

Your user code: ZSTN-HSMC
If a browser doesn't automatically open, please visit this URL from any trusted device: https://auth.classiq.io/activate?user_code=ZSTN-HSMC


In [9]:
#authenticate()

Please visit this URL from any trusted device to authenticate: https://auth.classiq.io/authorize?client_id=f6721qMOVoDAOVkzrv8YaWassRKSFX6Y&response_type=code&audience=https%3A%2F%2Fcadmium-be&redirect_uri=https%3A%2F%2Fauth.classiq.io%2Factivate%3Fuser_code%3DFMGH-LWVV&scope=offline_access
Your user code: FMGH-LWVV


# Two dimentional quantum walk

We use inner degree of freedom as 
$$
|\psi\rangle = \frac{1}{2}(|0\rangle + |1\rangle + |2\rangle + |3\rangle )
$$
Here, to implement flip-flop walk easily, we define the move direction as 
- $|0\rangle$ : Left
- $|1\rangle$ : Up
- $|2\rangle$ : Right
- $|3\rangle$ : Down

We can only use increment $+2$ to achieve flip-flop walk, i.e., $|0\rangle \xleftrightarrow{\text{+2\,\rm mod 4}} |2\rangle$, $|1\rangle \xleftrightarrow{\text{+2\,\rm mod 4}} |3\rangle$.

In [5]:
import matplotlib.pyplot as plt
import numpy as np
from classiq import *
import time

In [8]:
d = 3
q = 3
x_piece = 2**q
N = x_piece ** d

k = 6
nx = int(np.ceil(np.log2(x_piece)))
nl = int(np.ceil(np.log2(k)))
NUM_SAMPLES = 1000
print("N =",N , "nx =", nx, "nl =", nl )

N = 512 nx = 3 nl = 3


In [ ]:
class pos_reg(QStruct):
    x: QNum[nx]
    y: QNum[nx]
    z: QNum[nx]

@qfunc
def prepare_pos_state(pos: pos_reg):
    if x_piece == (2**nx):
        hadamard_transform(pos)
    else:
        prob_array = np.ones(2**nx)/x_piece
        prob_array[x_piece:2**(nx)] = 0
        inplace_prepare_state(prob_array.tolist(), 0.0, pos.x)
        inplace_prepare_state(prob_array.tolist(), 0.0, pos.y)
        inplace_prepare_state(prob_array.tolist(), 0.0, pos.z)

@qfunc 
def prepare_coin_state(l:QNum[nl]):
    if k == 2**nl:
        hadamard_transform(l)
    elif k < 2**nl:
        prob_list = [1/k, 1/k, 1/k, 1/k, 1/k, 1/k, 0, 0]
        inplace_prepare_state(prob_list, 0.0, l)

@qfunc 
def initial_state(pos: pos_reg, l:QNum):
    prepare_pos_state(pos)
    prepare_coin_state(l)

@qperm
def oracle_operator(pos: pos_reg):
    control(((pos.x == x_piece//2) & (pos.y == x_piece//2) & (pos.z == x_piece//2)), lambda: phase(np.pi))


@qfunc
def coin_opeator(l: QNum):
    grover_diffuser(lambda psi: prepare_coin_state(psi), l)

@qfunc
def shift_plus(p: QNum):
    p += 1 

@qfunc
def shift_minus(p: QNum):
    p += -1

@qfunc 
def move_opeartor(pos: pos_reg, l:QNum):
    #pos_list = [pos.x, pos.y, pos.z]
    # for i in range(d):
    #     control(l == i, lambda: shift_minus(pos_list[i]))
    #     control(l == d+i, lambda: shift_plus(pos_list[i]))
    control(l == 0, lambda: shift_minus(pos.x))
    control(l == 1, lambda: shift_plus(pos.x))
    control(l == 2, lambda: shift_minus(pos.y))
    control(l == 3, lambda: shift_plus(pos.y))
    control(l == 4, lambda: shift_minus(pos.z))
    control(l == 5, lambda: shift_plus(pos.z))

@qfunc 
def flip_operator(l:QArray):
    X(l[0])

@qfunc 
def shift_opeartor(pos: pos_reg, l:QNum):
    move_opeartor(pos, l)
    flip_operator(l)

@qfunc 
def quantum_walk(pos: pos_reg, l:QNum):
    coin_opeator(l)
    shift_opeartor(pos, l)

@qfunc 
def coin_walk(t: CInt, pos: pos_reg, l:QNum):
    power(
        t,
        lambda: [oracle_operator(pos),quantum_walk(pos, l)],
    )

#t = 1
@qfunc
def main(t: CInt, p: Output[pos_reg]):
    l = QNum()
    allocate(p.size,p)
    allocate(nl,l)
    initial_state(p,l)
    coin_walk(t, p, l)
    drop(l)


# @qfunc
# def main(pos: Output[pos_reg]):
#     l = QNum()
#     allocate(pos.size, pos)
#     allocate(2,l)
#     initial_state(pos,l)
#     t = int(np.sqrt(N) * np.log2(N))
#     coin_walk(t, pos, l)
#     drop(l)


In [23]:
qprog = synthesize(main)
#show(qprog)

In [47]:
# preferences = Preferences(
#     custom_hardware_settings=CustomHardwareSettings(basis_gates=["cx", "u"]),
#     transpilation_option="custom",
#     debug_mode=False,
#     #timeout_seconds=600
# )
# constraints = Constraints(max_depth=200, optimization_parameter="depth")
#
# qprog = synthesize(main, preferences=preferences,constraints=constraints)
# show(qprog)

Quantum program link: https://platform.classiq.io/circuit/390z0pQq6gTPU8GH92nWkEUXNGW


https://platform.classiq.io/circuit/390z0pQq6gTPU8GH92nWkEUXNGW?login=True&version=16

In [ ]:
# preferences = Preferences(
#     optimization_level=0, 
#     optimization_timeout_seconds=90, 
#     transpilation_option="none"
# )
# backend_preferences = ClassiqBackendPreferences(
#     backend_name=ClassiqSimulatorBackendNames.SIMULATOR_STATEVECTOR
# )
# execution_preferences = ExecutionPreferences(
#     num_shots=1, 
#     backend_preferences=backend_preferences
# )

# qmod = create_model(
#     main,
#     #constraints=constraints,
#     preferences=preferences,
#     execution_preferences=execution_preferences,
# )
# qprog = synthesize(qmod)

In [34]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

exec_pref = ExecutionPreferences(num_shots=NUM_SAMPLES)
with ExecutionSession(qprog,execution_preferences=exec_pref) as es:
    sample_result = es.sample({"t": 80})
    print(sample_result.dataframe)

     p.x  p.y  p.z  counts  probability  bitstring
0      4    4    4     313        0.313  100100100
1      5    4    4      67        0.067  100100101
2      4    4    5      64        0.064  101100100
3      4    3    4      61        0.061  100011100
4      4    4    3      59        0.059  011100100
..   ...  ...  ...     ...          ...        ...
190    6    4    7       1        0.001  111100110
191    6    5    7       1        0.001  111101110
192    3    6    7       1        0.001  111110011
193    4    6    7       1        0.001  111110100
194    7    7    7       1        0.001  111111111

[195 rows x 6 columns]


In [35]:
df = sample_result.dataframe
df.loc[(df["p.x"] == x_piece//2) & (df["p.y"] == x_piece//2)& (df["p.z"] == x_piece//2), "probability"]

0    0.313
Name: probability, dtype: float64

# 回路評価

## $d=3$ 立法格子

In [2]:
from classiq import *
from qiskit import QuantumCircuit
from qiskit import qasm3
from qiskit.compiler import transpile as qiskit_transpile

preferences_default = Preferences(
    timeout_seconds=2000
) # type: ignore
preferences_opt = Preferences(
    custom_hardware_settings=CustomHardwareSettings(basis_gates=["cx", "u"]),
    transpilation_option="custom",
    debug_mode=False,
    timeout_seconds=2000
) # type: ignore

preferences_faster = Preferences(
   debug_mode=False,
   optimization_level=0,
   transpilation_option="custom",
   output_format=["qasm"],
   qasm3=True,
)

/Users/reisato/PycharmProjects/pythonProject1/.venv/lib/python3.11/site-packages/classiq/__init__.py:108: UserWarning: 'nest_asyncio2' is not installed; falling back to the legacy 'nest_asyncio'. 'nest_asyncio2' ships with modern Jupyter (ipykernel >= 7.3); consider upgrading your Jupyter installation, for example: pip install --upgrade jupyter ipykernel
  enable_jupyter_notebook()


In [12]:
def quantum_walk_circuit_eval(t, x_piece, nx):

    class pos_reg(QStruct):
        x: QNum[nx]
        y: QNum[nx]
        z: QNum[nx]

    @qfunc
    def prepare_pos_state(pos: pos_reg):
        if x_piece == (2**nx):
            hadamard_transform(pos)
        else:
            prob_array = np.ones(2**nx)/x_piece
            prob_array[x_piece:2**(nx)] = 0
            inplace_prepare_state(prob_array.tolist(), 0.0, pos.x)
            inplace_prepare_state(prob_array.tolist(), 0.0, pos.y)
            inplace_prepare_state(prob_array.tolist(), 0.0, pos.z)

    @qfunc 
    def prepare_coin_state(l:QNum[nl]):
        if k == 2**nl:
            hadamard_transform(l)
        elif k < 2**nl:
            prob_list = [1/k, 1/k, 1/k, 1/k, 1/k, 1/k, 0, 0]
            inplace_prepare_state(prob_list, 0.0, l)

    @qfunc 
    def initial_state(pos: pos_reg, l:QNum):
        prepare_pos_state(pos)
        prepare_coin_state(l)

    @qperm
    def oracle_operator(pos: pos_reg):
        control(((pos.x == x_piece//2) & (pos.y == x_piece//2) & (pos.z == x_piece//2)), lambda: phase(np.pi))


    @qfunc
    def coin_opeator(l: QNum):
        grover_diffuser(lambda psi: prepare_coin_state(psi), l)

    @qfunc
    def shift_plus(p: QNum):
        p += 1 

    @qfunc
    def shift_minus(p: QNum):
        p += -1

    @qfunc 
    def move_opeartor(pos: pos_reg, l:QNum):
        control(l == 0, lambda: shift_minus(pos.x))
        control(l == 1, lambda: shift_plus(pos.x))
        control(l == 2, lambda: shift_minus(pos.y))
        control(l == 3, lambda: shift_plus(pos.y))
        control(l == 4, lambda: shift_minus(pos.z))
        control(l == 5, lambda: shift_plus(pos.z))

    @qfunc 
    def flip_operator(l:QArray):
        X(l[0])

    @qfunc 
    def shift_opeartor(pos: pos_reg, l:QNum):
        move_opeartor(pos, l)
        flip_operator(l)

    @qfunc 
    def quantum_walk(pos: pos_reg, l:QNum):
        coin_opeator(l)
        shift_opeartor(pos, l)

    @qfunc 
    def coin_walk(pos: pos_reg, l:QNum):
        power(
            t,
            lambda: [oracle_operator(pos),quantum_walk(pos, l)],
        )


    @qfunc
    def main(pos: Output[pos_reg]):
        l = QNum()
        allocate(pos.size, pos)
        allocate(nl,l)
        initial_state(pos,l)
        coin_walk(pos, l)
        drop(l)

        
    start_time = time.time()
    """
    without optimization
    """
    qprog = synthesize(main, preferences=preferences_default)
    """
    with optimization
    """
    #constraints = Constraints(optimization_parameter="cx", max_width=100)
    #qprog = synthesize(main, constraints=constraints, preferences=preferences_opt)
    """
    with faster compilation
    """
    #constraints = Constraints(max_width=100)
    #qprog = synthesize(main, constraints=constraints, preferences=preferences_faster)
    
    #show(qprog)
    
    compilation_time = time.time() - start_time
    width = qprog.data.width
    try:
        depth = qprog.transpiled_circuit.depth
        cx_counts = qprog.transpiled_circuit.count_ops["cx"]
    except Exception:
        qasm_str = qprog.transpiled_circuit.qasm
        qc       = qiskit_qasm3_import.parse(qasm_str)
        qc_t     = transpile(qc)
        depth     = qc_t.depth()
        cx_counts = qc_t.count_ops().get("cx", 0)
        #qc = qasm3.loads(qprog.qasm)
        #qc_t = qiskit_transpile(qc)
        #depth = qc_t.depth()
        #cx_counts = qc_t.count_ops().get("cx", 0)
        
    print(f"==== classiq for {NameError}==== time {compilation_time}")
    print(f"x_piece={x_piece}, nx={nx}, nl={nl},  N={N}, t_opt={t}, compile_time={compilation_time}, width={width}, depth={depth},cx={cx_counts}")
    return {
        "x_piece": x_piece,
        "nx": nx,
        "nl": nl,
        "N": N,
        "t_opt": t,
        "compile_time_sec": compilation_time,
        "width": width,
        "depth": depth,
        "cx_count": cx_counts,
    }


## 座標レジスタがアダマールゲートのみ

In [13]:
import pandas as pd

records = []
d = 3
k = 2*d
nl = int(np.ceil(np.log2(k)))
L_list = [2, 4, 8, 16, 32]#, 64, 128, 256, 512, 1024]#, 2048, 4096, 8192,16384, 32768, 65536, 131072, 262144]
#L_list = [16, 32, 64]#, 1024, 2048, 4096, 8192,16384, 32768, 65536, 131072, 262144]
for x_piece in L_list:
    N = x_piece**d
    t_opt = int(np.sqrt(N))
    nx = int(np.ceil(np.log2(x_piece)))
    result = quantum_walk_circuit_eval(t_opt, x_piece, nx)
    records.append(result)

==== classiq for <class 'NameError'>==== time 14.891231298446655
x_piece=2, nx=1, nl=3,  N=8, t_opt=2, compile_time=14.891231298446655, width=8, depth=558,cx=342
==== classiq for <class 'NameError'>==== time 29.486990213394165
x_piece=4, nx=2, nl=3,  N=64, t_opt=8, compile_time=29.486990213394165, width=13, depth=4804,cx=3554
==== classiq for <class 'NameError'>==== time 70.51712417602539
x_piece=8, nx=3, nl=3,  N=512, t_opt=22, compile_time=70.51712417602539, width=17, depth=17685,cx=13084
==== classiq for <class 'NameError'>==== time 173.91458296775818
x_piece=16, nx=4, nl=3,  N=4096, t_opt=64, compile_time=173.91458296775818, width=21, depth=67522,cx=48442
==== classiq for <class 'NameError'>==== time 745.0811829566956
x_piece=32, nx=5, nl=3,  N=32768, t_opt=181, compile_time=745.0811829566956, width=25, depth=249307,cx=180813


In [11]:
df = pd.DataFrame(records)
df


,x_piece,nx,nl,N,t_opt,compile_time_sec,width,depth,cx_count
0,2,1,3,8,2,19.885294,8,558,342
1,4,2,3,64,8,35.173258,13,4804,3554
2,8,3,3,512,22,71.856103,17,17685,13084
3,16,4,3,4096,64,173.861314,21,67522,48442


In [ ]:
df.to_csv("d3_quantum_walk_circuit_eval_without_optmization.csv", index=False)
#df.to_csv("d3_quantum_walk_circuit_eval_with_optmization.csv", index=False)

## 任意のノード数

In [64]:
records = []
d = 3
k = 2*d
nl = int(np.ceil(np.log2(k)))
L_list = [int(2**m) for m in np.linspace(1, 18, 20)]
for x_piece in L_list:
    N = x_piece**d
    t_opt = int(np.sqrt(N))
    nx = int(np.ceil(np.log2(x_piece)))
    result = quantum_walk_circuit_eval(t_opt, x_piece, nx)
    records.append(result)

==== classiq for <class 'NameError'>==== time 12.091266870498657
x_piece=2, nx=1, nl=3,  N=8, t_opt=2, compile_time=12.091266870498657, width=8, depth=554,cx=342
==== classiq for <class 'NameError'>==== time 13.422841787338257
x_piece=3, nx=2, nl=3,  N=27, t_opt=5, compile_time=13.422841787338257, width=13, depth=2936,cx=2135
==== classiq for <class 'NameError'>==== time 17.936836004257202
x_piece=6, nx=3, nl=3,  N=216, t_opt=14, compile_time=17.936836004257202, width=17, depth=9884,cx=7376
==== classiq for <class 'NameError'>==== time 23.996365070343018
x_piece=12, nx=4, nl=3,  N=1728, t_opt=41, compile_time=23.996365070343018, width=21, depth=43006,cx=30991
==== classiq for <class 'NameError'>==== time 47.99655294418335
x_piece=23, nx=5, nl=3,  N=12167, t_opt=110, compile_time=47.99655294418335, width=25, depth=147132,cx=103374
==== classiq for <class 'NameError'>==== time 120.79784321784973
x_piece=44, nx=6, nl=3,  N=85184, t_opt=291, compile_time=120.79784321784973, width=29, depth

ClassiqAPIError: The resources needed to execute this request are insufficient.
 This may be due to computational limitations, or high load on Classiq's servers.
 We suggest trying with alternative parameters, or reducing the resource consumption.


Error identifier: ED74EE1CD-F51D-4071-B485-837DBDD79C7B.
If you need further assistance, please reach out on our Community Slack channel at: https://short.classiq.io/join-slack or open a support ticket at: https://classiq-community.freshdesk.com/support/tickets/new

In [65]:
df = pd.DataFrame(records)
df

,x_piece,nx,nl,N,t_opt,compile_time_sec,width,depth,cx_count
0,2,1,3,8,2,12.091267,8,554,342
1,3,2,3,27,5,13.422842,13,2936,2135
2,6,3,3,216,14,17.936836,17,9884,7376
3,12,4,3,1728,41,23.996365,21,43006,30991
4,23,5,3,12167,110,47.996553,25,147132,103374
5,44,6,3,85184,291,120.797843,29,436309,322317
6,82,7,3,551368,742,304.247514,33,1359563,989458


In [66]:
df.to_csv("ver2_d3_quantum_walk_circuit_eval_without_optmization.csv", index=False)
#df.to_csv("ver2_d3_quantum_walk_circuit_eval_with_optmization.csv", index=False)

## Reference

https://arxiv.org/pdf/1010.4705